In [1]:
# Installing the HuggingFace PEFT (Parameter Efficient Fine-Tuning) library
# (purpose: enabling LoRA and other efficient training adapters)
%pip install peft==0.4.0

# Creating a directory to cache our outputs or temporary files
# (this is for keeping training outputs organized)
!mkdir cache

# Installing Hugging Face datasets library
# (to load the quote dataset easily)
!pip install datasets

mkdir: cannot create directory ‘cache’: File exists


In [2]:
# Loading the base model, tokenizer, and dataset:


# Importing required tools to load the model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# Setting the model name
model_name = "bigscience/bloomz-560m" # This is a relatively lightweight large language model (~560 million parameters)

# Loading the tokenizer (converts text into tokens (numbers) the model understands)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Loading the base pre-trained causal language model
# CausalLM = Causal Language Model, meaning it predicts the next word given previous words
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Loading the dataset from Hugging Face
# This line loads a 10% sample of the training split of the english_quotes dataset
# "split='train[:10%]'" means: take the first 10% of the training set
data = load_dataset("Abirate/english_quotes", split="train[:10%]")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Preprocessing the dataset

# Tokenizing the dataset
# Each quote in the dataset will be passed through the tokenizer
# batched=True lets the tokenizer process multiple quotes at once for efficiency
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

In [4]:
# (for me) Viewing the first few tokenized entries after preprocessing
train_sample = data.select(range(5))
print(train_sample)

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})


In [5]:
# Configuring LoRA (LoraConfig)

# Importing LoRA tools from the PEFT library
import peft
from peft import LoraConfig, get_peft_model

# Configuring LoRA
# LoraConfig defines how the adapter layers will be structured and applied
lora_config = LoraConfig(
    r=1,  # Rank of the low-rank matrices; controls how "small" the adapter layers are
         # Larger r means more capacity (slightly more compute/memory)

    lora_alpha=1,  # A scaling factor used during training; adjusts the impact of the adapter
                    # Rule of thumb: lora_alpha = 4 * r is a common starting point

    target_modules=["query_key_value"],
    # This is model-specific; for BloomZ, the relevant linear layers for attention are called "query_key_value"
    # You’re telling LoRA to attach adapters to these layers only (not all layers)

    lora_dropout=0.05,  # Adds regularization to prevent overfitting during training
                       # 0.1 is a standard starting point

    bias="none",  # We’re not training bias parameters, just the LoRA matrices

    task_type="CAUSAL_LM"  # This tells PEFT we’re fine-tuning a causal language model (text generation)
)

In [6]:
# Applying LoRA to the pre-trained model

# Adding LoRA adapter layers to the model
peft_model = get_peft_model(foundation_model, lora_config)

# Verifying which parameters are trainable
# This prints how many parameters are being updated (only a small number compared to full fine-tuning)
print(peft_model.print_trainable_parameters())

# Only a tiny fraction of parameters are trainable now, making it fast and resource-light

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.01757585078102687
None


In [7]:
# Training Setup — TrainingArguments and Trainer:
# Where to save the model
# What learning rate to use
# How many epochs to train
# Whether to use GPU or CPU

# 1- Defining training arguments:
# Importing training tools
from transformers import TrainingArguments, Trainer
import os

# Defining where to save outputs
output_directory = os.path.join("../cache/working", "peft_lab_outputs")

# Defining training settings
training_args = TrainingArguments(
    output_dir=output_directory,       # Directory to save model checkpoints and outputs
    auto_find_batch_size=True,         # Automatically pick a batch size that fits into memory
    learning_rate=3e-2,                # Higher learning rate for LoRA training (not full model)
    num_train_epochs=3,                # Number of passes through the training dataset
    use_cpu=True,                      # Use CPU for training (change to False if using GPU)
    report_to="none"                   # Disable reporting to external tools like WandB
)

In [8]:
# 2- Defining the Trainer

# Importing only the specific class needed for data collation
from transformers import DataCollatorForLanguageModeling

## Initializing the Trainer with corrected data collator
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # We're doing causal language modeling, not masked language modeling
    )
)

In [ ]:
# 3- # Start the training process
trainer.train()

Step,Training Loss


# I couldn't finish because impossible to train (not enough RAM)